In [1]:
# Load dataset
import json

with open("kz_dataset/manifest.json", "r") as f:
    manifest = json.load(f)

print(manifest)


{'sample_rate': 16000, 'channels': 1, 'items': [{'index': 0, 'text': 'Hello, how are you today?', 'wav': '000.wav', 'stt_text': 'Hello, how are you today?'}, {'index': 1, 'text': 'The quick brown fox jumps over the lazy dog.', 'wav': '001.wav', 'stt_text': 'The quick bro forced zoom over the lazy dock.'}, {'index': 2, 'text': 'Please open the window.', 'wav': '002.wav', 'stt_text': 'Please, open the window.'}, {'index': 3, 'text': 'I would like a cup of coffee.', 'wav': '003.wav', 'stt_text': 'I would like a cup of coffee.'}, {'index': 4, 'text': 'What time is the meeting?', 'wav': '004.wav', 'stt_text': 'What time is the misting?'}, {'index': 5, 'text': 'She sells seashells by the seashore.', 'wav': '005.wav', 'stt_text': "She's self, she's self by the R.C. show."}, {'index': 6, 'text': 'The weather is nice this afternoon.', 'wav': '006.wav', 'stt_text': 'The weather is nice this afternoon.'}, {'index': 7, 'text': 'Could you repeat that more slowly?', 'wav': '007.wav', 'stt_text': 'Co

In [2]:
import sounddevice as sd
import os
import tempfile
from collections.abc import Sequence

import numpy as np
import sounddevice as sd
import soundfile as sf
from forcealign import ForceAlign


SAMPLE_RATE = manifest['sample_rate']
CHANNELS = manifest['channels']
SD_DTYPE = "float32"
FOLDER_WAV = "kz_dataset"

def open_mic_input_stream(
    callback,
    *,
    sample_rate: int = SAMPLE_RATE,
    channels: int = CHANNELS,
    dtype: str = SD_DTYPE,
) -> sd.InputStream:
    """InputStream mặc định (mono, sample_rate). Gọi .start() sau khi tạo."""
    return sd.InputStream(
        channels=channels,
        samplerate=sample_rate,
        dtype=dtype,
        callback=callback,
    )


def force_align(wav_path: str, transcript: str):
    """Chạy ForceAlign, trả về danh sách Word từ forcealign."""
    aligner = ForceAlign(audio_file=wav_path, transcript=transcript)
    return aligner.inference()

for item in manifest["items"]:
    wav_path = os.path.join(FOLDER_WAV, item["wav"])
    transcript = item["stt_text"]
    words = force_align(wav_path, transcript)
    print(words)


[HELLO: 0.0 -- 0.887(s), HOW: 1.451 -- 1.592(s), ARE: 1.612 -- 1.693(s), YOU: 1.713 -- 1.854(s), TODAY: 1.894 -- 2.237(s)]
[THE: 0.0 -- 0.522(s), QUICK: 0.582 -- 0.863(s), BRO: 1.063 -- 1.384(s), FORCED: 1.605 -- 1.986(s), ZOOM: 2.548 -- 2.849(s), OVER: 3.09 -- 3.411(s), THE: 3.431 -- 3.672(s), LAZY: 3.973 -- 4.334(s), DOCK: 4.394 -- 4.695(s)]
[PLEASE: 0.0 -- 0.723(s), OPEN: 1.064 -- 1.406(s), THE: 1.466 -- 1.607(s), WINDOW: 1.667 -- 2.028(s)]
[I: 0.0 -- 0.645(s), WOULD: 0.705 -- 0.907(s), LIKE: 0.967 -- 1.169(s), A: 1.189 -- 1.229(s), CUP: 1.29 -- 1.491(s), OF: 1.531 -- 1.592(s), COFFEE: 1.672 -- 2.116(s)]
[WHAT: 0.0 -- 0.423(s), TIME: 0.504 -- 0.827(s), IS: 0.948 -- 1.109(s), THE: 1.634 -- 1.795(s), MISTING: 1.876 -- 2.279(s)]
[SHES: 0.0 -- 0.602(s), SELF: 0.702 -- 1.083(s), SHES: 1.905 -- 2.065(s), SELF: 2.105 -- 2.446(s), BY: 2.687 -- 3.007(s), THE: 3.268 -- 3.368(s), RC: 3.549 -- 3.749(s), SHOW: 4.05 -- 4.391(s)]
[THE: 0.0 -- 0.463(s), WEATHER: 0.503 -- 0.925(s), IS: 1.127 -- 1.22

In [3]:
import difflib

old = ['the', 'quick', 'is', 'bro', 'forced', 'zoom', 'over', 'the', 'lazy', 'dock']
new = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'today']


matcher = difflib.SequenceMatcher(None, old, new)

result = []

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    result.append({
        "old_text": " ".join(old[i1:i2]),
        "new_text": " ".join(new[j1:j2]),
        "type": tag
    })

print(result)

[{'old_text': 'the quick', 'new_text': 'the quick', 'type': 'equal'}, {'old_text': 'is bro forced zoom', 'new_text': 'brown fox jumps', 'type': 'replace'}, {'old_text': 'over the lazy', 'new_text': 'over the lazy', 'type': 'equal'}, {'old_text': 'dock', 'new_text': 'dog today', 'type': 'replace'}]


In [4]:
# Đã cài: pip install -e .  →  from kzen.utils import …
from kzen.utils import compare_text

for item in manifest["items"]:
    text = item["text"]
    stt_text = item["stt_text"]
    result = compare_text(stt_text=stt_text, ref_text=text)
    print(result)


diff_results=[DiffResult(ref_text='hello how are you today', stt_text='hello how are you today', type=<DiffType.EQUAL: 'equal'>)] iou_level=<IouLevel.EXCELLENT: 'excellent'> iou=100
diff_results=[DiffResult(ref_text='the quick', stt_text='the quick', type=<DiffType.EQUAL: 'equal'>), DiffResult(ref_text='brown fox jumps', stt_text='bro forced zoom', type=<DiffType.REPLACE: 'replace'>), DiffResult(ref_text='over the lazy', stt_text='over the lazy', type=<DiffType.EQUAL: 'equal'>), DiffResult(ref_text='dog', stt_text='dock', type=<DiffType.REPLACE: 'replace'>)] iou_level=<IouLevel.BAD: 'bad'> iou=33
diff_results=[DiffResult(ref_text='please open the window', stt_text='please open the window', type=<DiffType.EQUAL: 'equal'>)] iou_level=<IouLevel.EXCELLENT: 'excellent'> iou=100
diff_results=[DiffResult(ref_text='i would like a cup of coffee', stt_text='i would like a cup of coffee', type=<DiffType.EQUAL: 'equal'>)] iou_level=<IouLevel.EXCELLENT: 'excellent'> iou=100
diff_results=[DiffResult